# Fama-French Factors for Vietnam

Construct Vietnamese equivalents of the SMB (size) and HML (value) factors using the classic Fama-French 2x3 sort. Universe is HOSE main board, rebalanced monthly.

**Why this matters**: every quant strategy in Vietnam ultimately gets benchmarked against SMB and HML. Building them yourself — rather than borrowing US factors and praying — is the only way to know what's actually driving your alpha.

## Setup

Mocked monthly returns for ~200 HOSE names. In production: `dc.dataset('equity.hose.monthly_returns').to_pandas(start='2015-01-01')` and `dc.dataset('equity.fundamentals.monthly').to_pandas(...)`.

In [ ]:
import pandas as pd  # noqa
import numpy as np  # noqa
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)
plt.rcParams['figure.figsize'] = (11, 5)

## Step 1 — Build the universe

200 mock tickers, each with monthly returns from Jan 2018 to Dec 2024, plus time-varying market cap and P/B.

In [ ]:
n_names = 200
dates = pd.date_range('2018-01-31', '2024-12-31', freq='ME')
tickers = [f'T{i:03d}' for i in range(n_names)]

# Latent size and value tilts per name
size_tilt = rng.normal(0, 1, n_names)
val_tilt  = rng.normal(0, 1, n_names)

# Monthly market cap (trillion VND), drifts up over time with noise
base_mcap = np.exp(rng.normal(2.0, 1.5, n_names))  # 1–500 T VND ranges
mcap = pd.DataFrame(
    np.outer(np.linspace(1.0, 1.4, len(dates)), base_mcap)
        * np.exp(rng.normal(0, 0.04, (len(dates), n_names))),
    index=dates, columns=tickers,
)

# P/B around 1.5 with cross-sectional spread
pb = pd.DataFrame(
    np.exp(rng.normal(0.4, 0.5, (len(dates), n_names))),
    index=dates, columns=tickers,
)

# Returns: market beta + size + value loadings + idio
mkt = rng.normal(0.012, 0.06, len(dates))                 # market 1.2%/mo, 6% vol
rets = (
    np.outer(mkt, np.ones(n_names))
    + np.outer(rng.normal(0, 0.02, len(dates)), size_tilt) * 0.5  # size premium
    + np.outer(rng.normal(0, 0.02, len(dates)), val_tilt) * 0.5   # value premium
    + rng.normal(0, 0.05, (len(dates), n_names))                  # idio
)
rets = pd.DataFrame(rets, index=dates, columns=tickers)
rets.iloc[:3, :5].round(3)

## Step 2 — Form the 2x3 sort

At each month-end:

- **Size**: median split on market cap → Small (S) / Big (B)
- **Value**: 30/40/30 split on P/B → Low (L) / Medium (M) / High (H)

*High P/B = growth, Low P/B = value in Fama-French notation. We keep their naming convention.*

That gives 6 portfolios: SL, SM, SH, BL, BM, BH.

In [ ]:
def assign_buckets(row_mcap, row_pb):
    size = np.where(row_mcap <= row_mcap.median(), 'S', 'B')
    pb_q = row_pb.quantile([0.3, 0.7])
    val = np.where(row_pb <= pb_q.iloc[0], 'L',
          np.where(row_pb >= pb_q.iloc[1], 'H', 'M'))
    return pd.Series(size + val, index=row_mcap.index)

# Bucket at each month using current cap & P/B
buckets = pd.DataFrame(
    {d: assign_buckets(mcap.loc[d], pb.loc[d]) for d in dates}
).T
buckets.index = dates
buckets.iloc[:3, :8]

## Step 3 — Value-weight the 6 portfolios

Each portfolio's monthly return is the cap-weighted average of constituents using *prior-month* weights, so we don't peek.

In [ ]:
labels = ['SL', 'SM', 'SH', 'BL', 'BM', 'BH']
port_rets = pd.DataFrame(index=dates[1:], columns=labels, dtype=float)

for prev, curr in zip(dates[:-1], dates[1:]):
    b = buckets.loc[prev]
    w = mcap.loc[prev]
    r = rets.loc[curr]
    for lbl in labels:
        members = b[b == lbl].index
        if len(members) == 0:
            continue
        ww = w[members] / w[members].sum()
        port_rets.loc[curr, lbl] = (r[members] * ww).sum()

port_rets.head().round(3)

## Step 4 — Compute SMB and HML

Standard Fama-French construction:

```
SMB = (SL + SM + SH)/3  -  (BL + BM + BH)/3
HML = (SH + BH)/2       -  (SL + BL)/2
```

In [ ]:
smb = (port_rets[['SL', 'SM', 'SH']].mean(axis=1)
       - port_rets[['BL', 'BM', 'BH']].mean(axis=1)).rename('SMB')

hml = (port_rets[['SH', 'BH']].mean(axis=1)
       - port_rets[['SL', 'BL']].mean(axis=1)).rename('HML')

factors = pd.concat([smb, hml], axis=1).dropna()
factors.describe().round(4)

## Step 5 — Plot the factor time series

Monthly returns are noisy. We plot both the monthly stream and the cumulative compounding to see the drift.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

factors.plot(ax=axes[0], lw=1.2)
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_title('Vietnamese SMB & HML — monthly returns')
axes[0].set_ylabel('Monthly return')
axes[0].grid(alpha=0.3)

(1 + factors).cumprod().plot(ax=axes[1], lw=2)
axes[1].set_title('Cumulative factor returns (growth of 1 VND)')
axes[1].set_ylabel('Cumulative')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6 — Factor summary stats

Annualized premia, vol, Sharpe, and t-stat on the monthly mean. A t-stat above 2 is the academic bar for 'statistically real'.

In [ ]:
def factor_stats(r):
    ann = r.mean() * 12
    vol = r.std() * np.sqrt(12)
    sr  = ann / vol
    t   = r.mean() / (r.std() / np.sqrt(len(r)))
    return pd.Series({'Ann Premium': ann, 'Ann Vol': vol, 'Sharpe': sr, 't-stat': t})

factors.apply(factor_stats).T.round(3)

## Step 7 — Save factors for downstream use

Persist to CSV so other notebooks (momentum, low-vol, alpha regressions) can load them as benchmarks.

In [ ]:
from pathlib import Path
out = Path('../../data')
out.mkdir(exist_ok=True)
factors.to_csv(out / 'vn_smb_hml_monthly.csv')
print('Saved to', out / 'vn_smb_hml_monthly.csv')

## Next steps

- `02-momentum-12-1.ipynb` — add UMD/momentum to your factor set
- `03-low-vol.ipynb` — a fourth factor specific to emerging markets
- `02-equity-research/01-vn30-fundamentals.ipynb` — regress your screener returns on these factors